# Pandas — One Day Intensive
**NAVTTC · Data Analytics Using Python**

Everything in one session: create, load, inspect, select, filter, clean, transform.

---
**Rule:** Run every cell. Read every comment. Don't copy-paste — type the code.

| Section | What you learn |
|---|---|
| 1 | Import + Create Series & DataFrames |
| 2 | Load from CSV / Excel |
| 3 | Inspect |
| 4 | Select — columns, loc, iloc |
| 5 | Filter — boolean + query |
| 6 | Missing values |
| 7 | Duplicates |
| 8 | Data types |
| 9 | String cleaning |
| 10 | Rename, sort, new columns |
| 11 | Mini project |

In [1]:
import pandas as pd
import numpy as np

print('pandas:', pd.__version__)
print('numpy :', np.__version__)

pandas: 3.0.1
numpy : 2.4.3


---
## 1 · Series and DataFrames

**Series** = one column with a label index.  
**DataFrame** = a table — multiple Series sharing the same index.

In [2]:
# ── Series from a list ───────────────────────────────────────────────────────
cities = pd.Series(['Karachi', 'Lahore', 'Islamabad', 'Peshawar', 'Quetta'])
print(cities)
print('\ndtype:', cities.dtype)      # object = strings
print('shape:', cities.shape)        # (5,) — 5 rows, 1D
print('index:', cities.index)        # RangeIndex 0..4

0      Karachi
1       Lahore
2    Islamabad
3     Peshawar
4       Quetta
dtype: str

dtype: str
shape: (5,)
index: RangeIndex(start=0, stop=5, step=1)


In [3]:
# ── Series with a custom index ───────────────────────────────────────────────
# Label-based index — useful when values have natural keys
prices = pd.Series(
    [250, 180, 420, 95, 310],
    index=['Rice', 'Flour', 'Oil', 'Sugar', 'Milk'],
    name='Price_PKR'
)
print(prices)
print('\nFlour costs:', prices['Flour'])   # label access
print('First item:', prices.iloc[0])       # position access

Rice     250
Flour    180
Oil      420
Sugar     95
Milk     310
Name: Price_PKR, dtype: int64

Flour costs: 180
First item: 250


In [4]:
# ── DataFrame from a dictionary ──────────────────────────────────────────────
# Each key = column name, each value = list of column values
# All lists MUST be the same length

df = pd.DataFrame({
    'name'   : ['Ahmed', 'Fatima', 'Hassan', 'Zara', 'Bilal', 'Sara'],
    'city'   : ['Karachi', 'Lahore', 'Islamabad', 'Karachi', 'Lahore', 'Quetta'],
    'score'  : [82, 91, 74, 88, 65, 79],
    'salary' : [45000, 62000, 38000, 55000, 41000, 58000],
    'dept'   : ['Sales', 'Tech', 'HR', 'Tech', 'Sales', 'HR']
})

print(df)

     name       city  score  salary   dept
0   Ahmed    Karachi     82   45000  Sales
1  Fatima     Lahore     91   62000   Tech
2  Hassan  Islamabad     74   38000     HR
3    Zara    Karachi     88   55000   Tech
4   Bilal     Lahore     65   41000  Sales
5    Sara     Quetta     79   58000     HR


In [5]:
# ── What a DataFrame IS ──────────────────────────────────────────────────────
# Each column is independently accessible as a Series

print(type(df['name']))          # pandas Series
print(type(df[['name','city']])) # pandas DataFrame (double brackets = multiple columns)

# This distinction matters — many methods work only on DataFrames, not Series

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


---
## 2 · Load from Files

In real work, you never type data manually. You load it from files.

In [6]:
# ── Create sample CSV files for this session ─────────────────────────────────
import os

# Sales data (messy on purpose — we'll clean it)
sales_csv = """order_id,customer,city,product,qty,unit_price,order_date
101,Ahmed Khan,Karachi ,Laptop,1,85000,2024-01-15
102,fatima ali,lahore,Mouse,3, 1200,2024-01-16
103,Hassan Raza,ISLAMABAD,Keyboard,2,3500,2024-01-16
104,Zara Malik,karachi ,Monitor,1,32000,2024-01-17
105,Ahmed Khan,Karachi ,Laptop,1,85000,2024-01-15
106,,Lahore,Headphones,2,4500,2024-01-18
107,Bilal Ahmed,Peshawar,Mouse,1,1200,2024-01-18
108,Sara Qureshi,Quetta,Keyboard,3,3500,
109,Nadia Shah,Lahore,Monitor,1,32000,2024-01-19
110,Omar Farooq,Karachi ,Laptop,2,85000,2024-01-20
"""

with open('sales.csv', 'w') as f:
    f.write(sales_csv)

print('sales.csv created')

sales.csv created


In [7]:
# ── pd.read_csv() — most important parameters ────────────────────────────────

sales = pd.read_csv('sales.csv')

print(sales)

   order_id      customer       city     product  qty  unit_price  order_date
0       101    Ahmed Khan   Karachi       Laptop    1       85000  2024-01-15
1       102    fatima ali     lahore       Mouse    3        1200  2024-01-16
2       103   Hassan Raza  ISLAMABAD    Keyboard    2        3500  2024-01-16
3       104    Zara Malik   karachi      Monitor    1       32000  2024-01-17
4       105    Ahmed Khan   Karachi       Laptop    1       85000  2024-01-15
5       106           NaN     Lahore  Headphones    2        4500  2024-01-18
6       107   Bilal Ahmed   Peshawar       Mouse    1        1200  2024-01-18
7       108  Sara Qureshi     Quetta    Keyboard    3        3500         NaN
8       109    Nadia Shah     Lahore     Monitor    1       32000  2024-01-19
9       110   Omar Farooq   Karachi       Laptop    2       85000  2024-01-20


In [8]:
# ── parse_dates — read dates as datetime, not strings ───────────────────────

sales = pd.read_csv('sales.csv', parse_dates=['order_date'])

print(sales.dtypes)   # order_date should now be datetime64 not object

order_id               int64
customer                 str
city                     str
product                  str
qty                    int64
unit_price             int64
order_date    datetime64[us]
dtype: object


In [9]:
# ── Writing back to file ─────────────────────────────────────────────────────

# index=False is almost always what you want
# Without it, Pandas writes the row numbers (0,1,2...) as an extra column

sales.to_csv('sales_export.csv', index=False, encoding='utf-8')
print('Saved sales_export.csv')

# Excel:
# sales.to_excel('sales_export.xlsx', index=False, sheet_name='Sales')

# JSON:
# sales.to_json('sales_export.json', orient='records', indent=2)

Saved sales_export.csv


---
## 3 · Inspect

First thing you do with any dataset — every time.

In [10]:
# ── Shape, columns, types ────────────────────────────────────────────────────
print('Shape  :', sales.shape)        # (rows, cols)
print('Columns:', list(sales.columns))
print()
print(sales.dtypes)                   # data type of each column

Shape  : (10, 7)
Columns: ['order_id', 'customer', 'city', 'product', 'qty', 'unit_price', 'order_date']

order_id               int64
customer                 str
city                     str
product                  str
qty                    int64
unit_price             int64
order_date    datetime64[us]
dtype: object


In [11]:
# ── info() — the most useful single command ───────────────────────────────────
# Shows: column names, non-null counts, dtypes, memory usage
# Non-null count < total rows = missing values in that column

sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    10 non-null     int64         
 1   customer    9 non-null      str           
 2   city        10 non-null     str           
 3   product     10 non-null     str           
 4   qty         10 non-null     int64         
 5   unit_price  10 non-null     int64         
 6   order_date  9 non-null      datetime64[us]
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 692.0 bytes


In [12]:
# ── describe() — statistics for numerical columns ───────────────────────────
# count, mean, std, min, 25%, 50%, 75%, max

sales.describe()

,order_id,qty,unit_price,order_date
count,10.00000,10.000000,10.000000,9
mean,105.50000,1.700000,33290.000000,2024-01-17 02:40:00
min,101.00000,1.000000,1200.000000,2024-01-15 00:00:00
25%,103.25000,1.000000,3500.000000,2024-01-16 00:00:00
50%,105.50000,1.500000,18250.000000,2024-01-17 00:00:00
75%,107.75000,2.000000,71750.000000,2024-01-18 00:00:00
max,110.00000,3.000000,85000.000000,2024-01-20 00:00:00
std,3.02765,0.823273,37547.583145,NaN


In [13]:
# ── head / tail / sample ─────────────────────────────────────────────────────
print('--- First 3 rows ---')
print(sales.head(3))

print('\n--- Last 2 rows ---')
print(sales.tail(2))

print('\n--- Random 3 rows ---')
print(sales.sample(3, random_state=42))  # random_state for reproducibility

--- First 3 rows ---
   order_id     customer       city   product  qty  unit_price order_date
0       101   Ahmed Khan   Karachi     Laptop    1       85000 2024-01-15
1       102   fatima ali     lahore     Mouse    3        1200 2024-01-16
2       103  Hassan Raza  ISLAMABAD  Keyboard    2        3500 2024-01-16

--- Last 2 rows ---
   order_id     customer      city  product  qty  unit_price order_date
8       109   Nadia Shah    Lahore  Monitor    1       32000 2024-01-19
9       110  Omar Farooq  Karachi    Laptop    2       85000 2024-01-20

--- Random 3 rows ---
   order_id    customer    city     product  qty  unit_price order_date
8       109  Nadia Shah  Lahore     Monitor    1       32000 2024-01-19
1       102  fatima ali  lahore       Mouse    3        1200 2024-01-16
5       106         NaN  Lahore  Headphones    2        4500 2024-01-18


In [16]:
# ── Missing value count — run this on every new dataset ─────────────────────
print('Missing values per column:')
print(sales.isnull().sum())

print(f'\nTotal missing cells: {sales.isnull().sum().sum()}')
print(f'Missing %: {(sales.isnull().sum().sum() / sales.size * 100):.1f}%')

Missing values per column:
order_id      0
customer      1
city          0
product       0
qty           0
unit_price    0
order_date    1
dtype: int64

Total missing cells: 2
Missing %: 2.9%


---
## 4 · Select — Columns, loc, iloc

Three ways to select data. You need all three — they're for different situations.

In [19]:
# ── Column selection ─────────────────────────────────────────────────────────

# Single column → returns a Series
print(sales['customer'])            # bracket notation — always works
print()

# Multiple columns → returns a DataFrame
print(sales[['customer', 'city']])

0      Ahmed Khan
1      fatima ali
2     Hassan Raza
3      Zara Malik
4      Ahmed Khan
5             NaN
6     Bilal Ahmed
7    Sara Qureshi
8      Nadia Shah
9     Omar Farooq
Name: customer, dtype: str

       customer       city
0    Ahmed Khan   Karachi 
1    fatima ali     lahore
2   Hassan Raza  ISLAMABAD
3    Zara Malik   karachi 
4    Ahmed Khan   Karachi 
5           NaN     Lahore
6   Bilal Ahmed   Peshawar
7  Sara Qureshi     Quetta
8    Nadia Shah     Lahore
9   Omar Farooq   Karachi 


In [20]:
# ── loc[] — label-based selection ───────────────────────────────────────────
# Syntax: df.loc[row_selector, column_selector]
# Row selector can be: index label, list of labels, boolean mask, or slice
# Column selector can be: column name, list of names, or slice

# Select row by index label
print('Row 2:')
print(df.loc[2])               # returns the row as a Series

print('\nRows 1 to 3, columns name and salary:')
print(df.loc[1:3, ['name', 'salary']])   # INCLUSIVE — loc includes the end

Row 2:
name         Hassan
city      Islamabad
score            74
salary        38000
dept             HR
Name: 2, dtype: object

Rows 1 to 3, columns name and salary:
     name  salary
1  Fatima   62000
2  Hassan   38000
3    Zara   55000


In [21]:
# ── iloc[] — integer position-based selection ────────────────────────────────
# Works exactly like Python list slicing — EXCLUSIVE end
# iloc ignores the index completely — always uses 0,1,2...

print('First 3 rows, first 3 columns:')
print(df.iloc[0:3, 0:3])   # rows 0,1,2 — columns 0,1,2 — end is exclusive

print('\nLast row:')
print(df.iloc[-1])         # -1 = last row, like Python lists

First 3 rows, first 3 columns:
     name       city  score
0   Ahmed    Karachi     82
1  Fatima     Lahore     91
2  Hassan  Islamabad     74

Last row:
name        Sara
city      Quetta
score         79
salary     58000
dept          HR
Name: 5, dtype: object


In [22]:
# ── loc vs iloc — the key difference ────────────────────────────────────────
#
#   loc[1:3]  → rows with INDEX LABELS 1, 2, 3   (inclusive)
#   iloc[1:3] → rows at POSITIONS     1, 2       (exclusive end, like Python)
#


# Demonstrate: filter df, then compare loc vs iloc
subset = df[df['dept'] == 'Tech'].copy()
print('Subset (Tech dept):')
print(subset)
print('\nsubset.iloc[0]  → first ROW by position:')
print(subset.iloc[0])

Subset (Tech dept):
     name     city  score  salary  dept
1  Fatima   Lahore     91   62000  Tech
3    Zara  Karachi     88   55000  Tech

subset.iloc[0]  → first ROW by position:
name      Fatima
city      Lahore
score         91
salary     62000
dept        Tech
Name: 1, dtype: object


---
## 5 · Filter — Boolean Masking and Query

Filtering is how you answer questions like: *"show me all Karachi customers with salary above 50,000"*

In [ ]:
# ── Boolean mask — how filtering actually works ──────────────────────────────
# Step 1: a condition produces a Series of True/False
mask = df['salary'] > 50000
print('Mask:')
print(mask)

print('\nFiltered rows:')
print(df[mask])   # Step 2: pass the mask to df[]

In [23]:
# ── Combine conditions with & (and) | (or) ~ (not) ──────────────────────────
# IMPORTANT: Use & and | NOT 'and' / 'or' — and each condition needs parentheses

# Karachi employees earning above 50,000
result = df[(df['city'] == 'Karachi') & (df['salary'] > 50000)]
print('Karachi + salary > 50k:')
print(result)

# Tech or HR departments
result2 = df[(df['dept'] == 'Tech') | (df['dept'] == 'HR')]
print('\nTech or HR:')
print(result2)

Karachi + salary > 50k:
   name     city  score  salary  dept
3  Zara  Karachi     88   55000  Tech

Tech or HR:
     name       city  score  salary  dept
1  Fatima     Lahore     91   62000  Tech
2  Hassan  Islamabad     74   38000    HR
3    Zara    Karachi     88   55000  Tech
5    Sara     Quetta     79   58000    HR


In [24]:
# ── isin() — cleaner than multiple OR conditions ─────────────────────────────

# Instead of: df[city == 'Karachi' | city == 'Lahore' | city == 'Quetta']
result = df[df['city'].isin(['Karachi', 'Lahore', 'Quetta'])]
print(result)

     name     city  score  salary   dept
0   Ahmed  Karachi     82   45000  Sales
1  Fatima   Lahore     91   62000   Tech
3    Zara  Karachi     88   55000   Tech
4   Bilal   Lahore     65   41000  Sales
5    Sara   Quetta     79   58000     HR


In [ ]:
# ── query() — SQL-like readable syntax ──────────────────────────────────────
# Good for complex conditions — easier to read than chained boolean masks

result = df.query("city == 'Karachi' and salary > 40000")
print(result)

# Use @variable_name to reference Python variables inside query
min_salary = 50000
result2 = df.query("salary >= @min_salary")
print(result2)

---
## 6 · Missing Values

Real data always has missing values. You have two options: drop them or fill them. Choose based on context.

In [ ]:
# ── Work with the sales DataFrame (has real missing values) ──────────────────
print('Missing values:')
print(sales.isnull().sum())
print()

# Show which rows have ANY missing value
print('Rows with missing data:')
print(sales[sales.isnull().any(axis=1)])

In [25]:
# ── dropna() — remove rows or columns with missing values ───────────────────

# Drop rows where ANY column has NaN (most common use)
clean = sales.dropna()
print(f'Original: {len(sales)} rows | After dropna: {len(clean)} rows')

# Drop rows only if ALL columns are NaN
clean2 = sales.dropna(how='all')

# Drop rows only if at least 3 values are missing
# clean3 = sales.dropna(thresh=len(sales.columns) - 2)

# Drop rows missing in a specific column only
clean4 = sales.dropna(subset=['customer'])
print(f'After dropna on customer only: {len(clean4)} rows')
print(clean)
print('\nCleaned data (customer column only):')
print(clean4)

Original: 10 rows | After dropna: 8 rows
After dropna on customer only: 9 rows
   order_id     customer       city   product  qty  unit_price order_date
0       101   Ahmed Khan   Karachi     Laptop    1       85000 2024-01-15
1       102   fatima ali     lahore     Mouse    3        1200 2024-01-16
2       103  Hassan Raza  ISLAMABAD  Keyboard    2        3500 2024-01-16
3       104   Zara Malik   karachi    Monitor    1       32000 2024-01-17
4       105   Ahmed Khan   Karachi     Laptop    1       85000 2024-01-15
6       107  Bilal Ahmed   Peshawar     Mouse    1        1200 2024-01-18
8       109   Nadia Shah     Lahore   Monitor    1       32000 2024-01-19
9       110  Omar Farooq   Karachi     Laptop    2       85000 2024-01-20

Cleaned data (customer column only):
   order_id      customer       city   product  qty  unit_price order_date
0       101    Ahmed Khan   Karachi     Laptop    1       85000 2024-01-15
1       102    fatima ali     lahore     Mouse    3        1200 202

In [ ]:
# ── fillna() — fill missing values ──────────────────────────────────────────

sales_filled = sales.copy()   # always work on a copy first

# Fill with a fixed value
sales_filled['customer'] = sales_filled['customer'].fillna('Unknown')

# Fill with column mean (only for numeric columns)
sales_filled['unit_price'] = sales_filled['unit_price'].fillna(
    sales_filled['unit_price'].mean()
)

# Forward fill — use previous valid value (useful for time series)
sales_filled['order_date'] = sales_filled['order_date'].ffill()

print('After filling:')
print(sales_filled.isnull().sum())

In [ ]:
# ── When to drop vs fill ─────────────────────────────────────────────────────
#
#  DROP when:
#   - Missing data is random and few rows affected
#   - The key identifying column (customer name, order_id) is missing
#
#  FILL when:
#   - Missing numeric → fill with mean or median
#   - Missing category → fill with mode or 'Unknown'
#   - Time series → forward fill makes logical sense
#
#  NEVER blindly drop all NaN rows on large datasets — check first

print('Decision made: fill customer=Unknown, drop rows missing order_date')
sales_clean = sales.copy()
sales_clean['customer'] = sales_clean['customer'].fillna('Unknown')
sales_clean = sales_clean.dropna(subset=['order_date'])
print(sales_clean)

---
## 7 · Duplicates

In [ ]:
# ── Find duplicates ──────────────────────────────────────────────────────────
# duplicated() returns True for rows that are duplicates

print('Duplicate rows in sales:')
print(sales[sales.duplicated()])          # shows second occurrence

print('\nAll occurrences of duplicates:')
print(sales[sales.duplicated(keep=False)]) # keep=False marks ALL duplicates

In [ ]:
# ── Remove duplicates ────────────────────────────────────────────────────────

# Keep first occurrence, drop the rest
sales_dedup = sales.drop_duplicates()
print(f'Before: {len(sales)} rows | After: {len(sales_dedup)} rows')

# Duplicate based on specific columns only
# (same customer + same product on same date = duplicate order)
sales_dedup2 = sales.drop_duplicates(
    subset=['customer', 'product', 'order_date'],
    keep='first'
)
print(f'Dedup on key columns: {len(sales_dedup2)} rows')

---
## 8 · Data Types

Wrong data type is the most common source of silent errors. A salary stored as a string cannot be averaged.

In [ ]:
# ── Create a DataFrame with type problems ────────────────────────────────────
# (This happens regularly when loading CSV files)

messy = pd.DataFrame({
    'name'    : ['Ahmed', 'Fatima', 'Hassan', 'Zara'],
    'salary'  : ['45000', '62000', 'N/A', '55000'],   # stored as strings!
    'age'     : ['28', '32', '25', '29'],              # strings instead of int
    'joined'  : ['2022-03-15', '2021-07-01', '2023-01-10', '2022-11-20'],  # string dates
    'active'  : ['True', 'True', 'False', 'True'],    # string booleans
})

print('Dtypes before fixing:')
print(messy.dtypes)      # everything is object

In [ ]:
# ── astype() — convert when data is clean ────────────────────────────────────
fixed = messy.copy()

fixed['age'] = fixed['age'].astype(int)
print('age after astype(int):', fixed['age'].dtype)

In [ ]:
# ── pd.to_numeric() — when data has errors ───────────────────────────────────
# astype() raises an error on 'N/A'
# to_numeric with errors='coerce' converts bad values to NaN instead of crashing

fixed['salary'] = pd.to_numeric(fixed['salary'], errors='coerce')
print('salary after to_numeric:')
print(fixed['salary'])   # 'N/A' becomes NaN, rest become float64

In [ ]:
# ── pd.to_datetime() — convert string dates ──────────────────────────────────
fixed['joined'] = pd.to_datetime(fixed['joined'])
print('joined dtype:', fixed['joined'].dtype)   # datetime64

# Now you can extract parts of the date
fixed['join_year']  = fixed['joined'].dt.year
fixed['join_month'] = fixed['joined'].dt.month
print(fixed[['name', 'joined', 'join_year', 'join_month']])

In [ ]:
# ── Category dtype — for low-cardinality string columns ─────────────────────
# If a column has few unique values (city, dept, status), 'category' uses ~10x less memory

df['dept'] = df['dept'].astype('category')
print('dept dtype:', df['dept'].dtype)
print('Categories:', df['dept'].cat.categories.tolist())

---
## 9 · String Cleaning

The `.str` accessor lets you apply string methods to every value in a column at once — no loops needed.

In [ ]:
# ── The city column in sales is messy ─────────────────────────────────────────
# 'Karachi ', 'lahore', 'ISLAMABAD' — inconsistent case and trailing spaces

print('Raw city values:')
print(sales['city'].unique())

print('\nValue counts (shows duplicates due to inconsistency):')
print(sales['city'].value_counts())

In [ ]:
# ── Standard cleaning chain ──────────────────────────────────────────────────
# Always apply in this order: strip → lower or title → then specific fixes

sales_clean = sales.copy()

sales_clean['city'] = (
    sales_clean['city']
    .str.strip()          # remove leading/trailing whitespace
    .str.title()          # Title Case — 'karachi' → 'Karachi'
)

sales_clean['customer'] = (
    sales_clean['customer']
    .fillna('Unknown')    # handle NaN before string ops
    .str.strip()
    .str.title()
)

print('Clean city values:')
print(sales_clean['city'].value_counts())

In [ ]:
# ── Other useful .str methods ────────────────────────────────────────────────

s = pd.Series(['  Ahmed Khan  ', 'FATIMA ALI', 'hassan raza', 'Zara-Malik'])

print('strip + lower:  ', s.str.strip().str.lower().tolist())
print('replace hyphen: ', s.str.replace('-', ' ', regex=False).tolist())
print('contains "ali": ', s.str.lower().str.contains('ali').tolist())
print('starts with A:  ', s.str.strip().str.startswith('A').tolist())
print('length:         ', s.str.strip().str.len().tolist())

In [ ]:
# ── split() and extract specific parts ──────────────────────────────────────

names = pd.Series(['Ahmed Khan', 'Fatima Ali', 'Hassan Raza'])

# Split into first and last name
split = names.str.split(' ', expand=True)   # expand=True → DataFrame
split.columns = ['first_name', 'last_name']
print(split)

---
## 10 · Rename, Sort, New Columns

In [ ]:
# ── Rename columns ───────────────────────────────────────────────────────────
# rename() takes a dict: {'old_name': 'new_name'}

df_renamed = df.rename(columns={
    'name'   : 'employee_name',
    'salary' : 'monthly_salary_pkr',
    'dept'   : 'department'
})
print(df_renamed.columns.tolist())

In [ ]:
# ── Sort values ──────────────────────────────────────────────────────────────

# Sort by single column descending
print('Sorted by salary (highest first):')
print(df.sort_values('salary', ascending=False))

# Sort by multiple columns
print('\nSorted by dept then salary:')
print(df.sort_values(['dept', 'salary'], ascending=[True, False]))

In [ ]:
# ── Add new columns — vectorised operations ──────────────────────────────────
# Never use a loop. Operations on whole columns run on NumPy arrays — fast.

df_work = df.copy()

# Annual salary
df_work['annual_salary'] = df_work['salary'] * 12

# Bonus — 10% for score above 85, 5% for others
# np.where(condition, value_if_true, value_if_false)
df_work['bonus_pct'] = np.where(df_work['score'] > 85, 0.10, 0.05)
df_work['bonus_pkr'] = df_work['salary'] * df_work['bonus_pct']

# Performance category using pd.cut
df_work['grade'] = pd.cut(
    df_work['score'],
    bins=[0, 60, 75, 90, 100],
    labels=['D', 'C', 'B', 'A']
)

print(df_work[['name', 'salary', 'annual_salary', 'score', 'bonus_pkr', 'grade']])

In [ ]:
# ── replace() — fix specific values in a column ──────────────────────────────

df_work['city'] = df_work['city'].replace({
    'Karachi' : 'KHI',
    'Lahore'  : 'LHR',
})
print(df_work[['name', 'city']])

---
## 11 · Mini Project — Clean a Real Dataset

Apply everything you learned. The `sales` dataset has: mixed case cities, trailing spaces in customer names, one missing customer, one missing order_date, and one duplicate order.

In [ ]:
# ── Step 1: Load fresh ────────────────────────────────────────────────────────
raw = pd.read_csv('sales.csv', parse_dates=['order_date'])
print('Raw shape:', raw.shape)
print(raw.dtypes)
print()
raw.info()

In [ ]:
# ── Step 2: Inspect problems ──────────────────────────────────────────────────
print('Missing values:')
print(raw.isnull().sum())

print('\nDuplicate rows:')
print(raw[raw.duplicated(keep=False)])

print('\nUnique cities (raw):')
print(sorted(raw['city'].dropna().unique()))

In [ ]:
# ── Step 3: Clean ────────────────────────────────────────────────────────────
clean = raw.copy()

# 3a. Fix strings
clean['city']     = clean['city'].str.strip().str.title()
clean['customer'] = clean['customer'].str.strip().str.title()
clean['product']  = clean['product'].str.strip().str.title()

# 3b. Fix unit_price — strip spaces and convert
clean['unit_price'] = pd.to_numeric(
    clean['unit_price'].astype(str).str.strip(), errors='coerce'
)

# 3c. Handle missing values
clean['customer'] = clean['customer'].fillna('Unknown')
clean = clean.dropna(subset=['order_date'])   # drop rows missing date

# 3d. Remove duplicates
clean = clean.drop_duplicates()

print('Clean shape:', clean.shape)
print()
print(clean)

In [ ]:
# ── Step 4: Add calculated column ────────────────────────────────────────────
clean['total_revenue'] = clean['qty'] * clean['unit_price']

print('Revenue by city:')
print(clean.groupby('city')['total_revenue'].sum().sort_values(ascending=False))

print('\nRevenue by product:')
print(clean.groupby('product')['total_revenue'].sum().sort_values(ascending=False))

In [ ]:
# ── Step 5: Export the clean file ────────────────────────────────────────────
clean.to_csv('sales_clean.csv', index=False)
print('Saved: sales_clean.csv')
print(f'Raw rows: {len(raw)} → Clean rows: {len(clean)}')
print(f'Columns in clean file: {list(clean.columns)}')

---
## Summary — What You Can Now Do

| Task | Code |
|---|---|
| Load CSV | `pd.read_csv('file.csv', parse_dates=['date'])` |
| Inspect | `df.info()`, `df.describe()`, `df.isnull().sum()` |
| Select column | `df['col']` or `df[['c1','c2']]` |
| Select rows by label | `df.loc[0:5, ['name','city']]` |
| Select rows by position | `df.iloc[0:5, 0:3]` |
| Filter | `df[df['salary'] > 50000]` |
| Multi-condition filter | `df[(df['city']=='Karachi') & (df['score']>80)]` |
| Drop missing | `df.dropna(subset=['customer'])` |
| Fill missing | `df['col'].fillna(df['col'].mean())` |
| Remove duplicates | `df.drop_duplicates()` |
| Fix data type | `df['col'].astype(int)` / `pd.to_numeric(errors='coerce')` |
| Clean strings | `df['col'].str.strip().str.title()` |
| Rename columns | `df.rename(columns={'old':'new'})` |
| Sort | `df.sort_values('col', ascending=False)` |
| New column | `df['new'] = df['a'] * df['b']` |
| Export | `df.to_csv('file.csv', index=False)` |

---
**Next session:** GroupBy, aggregation, pivot tables, and merging DataFrames.